# Retail Analytics ETL Pipeline Demo

This notebook walks through the entire ETL process:
1. **Data Source** — where data comes from (Kaggle)
2. **Load** — read raw CSV into pandas
3. **Clean & Transform** — normalize columns, derive metrics
4. **Output & Visualize** — show results and key insights

## 1. Data Source

**Where does the data come from?**

- **Dataset:** `ankitbansal06/retail-orders` from Kaggle
- **Download:** `retail_analysis.py` downloads via Kaggle API when credentials are present
- **Output:** saved to `data/orders.csv` (9,994 rows of retail transactions)
- **Columns:** Order ID, Date, Ship Mode, Segment, Country, City, State, Postal Code, Region, Category, Sub-Category, Product ID, Cost Price, List Price, Quantity, Discount %

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Path to raw data (from Kaggle download)
raw_csv = '../data/orders.csv'

# Load raw data
df_raw = pd.read_csv(raw_csv, na_values=['Not Available', 'unknown'])

print('RAW DATA SNAPSHOT')
print('=' * 80)
print(f'Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
print(f'\nFirst 3 rows:')
print(df_raw.head(3))

## 2. Clean & Transform

The raw data undergoes several transformations:
- **Column names:** convert to lowercase, replace spaces with underscores
- **Derived metrics:** discount amount, sale price, profit
- **Date conversion:** order_date to datetime
- **Drop intermediates:** remove list_price, cost_price, discount_percent after use

In [ ]:
# Step 1: Normalize column names
df = df_raw.copy()
print('STEP 1: Normalize column names')
print(f'Before: {list(df.columns[:5])}...')

df.columns = df.columns.str.lower().str.replace(' ', '_')

print(f'After:  {list(df.columns[:5])}...')
print()

In [ ]:
# Step 2: Derive discount, sale_price, profit
print('STEP 2: Derive new columns')
print(f'Available columns: {list(df.columns)}')
print()

if {'list_price', 'discount_percent'}.issubset(df.columns):
    df['discount'] = df['list_price'] * df['discount_percent'] * 0.01
    df['sale_price'] = df['list_price'] - df['discount']
    print('✓ Created: discount = list_price * discount_percent * 0.01')
    print('✓ Created: sale_price = list_price - discount')
else:
    print('⚠ Columns list_price or discount_percent missing')

if {'sale_price', 'cost_price'}.issubset(df.columns):
    df['profit'] = df['sale_price'] - df['cost_price']
    print('✓ Created: profit = sale_price - cost_price')
else:
    print('⚠ Columns sale_price or cost_price missing')

print()
print('Example row with derived columns:')
cols_to_show = ['order_id', 'list_price', 'discount_percent', 'discount', 'sale_price', 'cost_price', 'profit']
print(df[cols_to_show].head(1).to_string())

In [ ]:
# Step 3: Convert order_date to datetime
print('STEP 3: Convert order_date to datetime')
print(f'Before: {df["order_date"].dtype}')

df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

print(f'After:  {df["order_date"].dtype}')
print(f'Date range: {df["order_date"].min()} to {df["order_date"].max()}')
print()

In [ ]:
# Step 4: Drop intermediate columns
print('STEP 4: Drop intermediate columns')
drop_cols = [c for c in ['list_price', 'cost_price', 'discount_percent'] if c in df.columns]
print(f'Dropping: {drop_cols}')

df.drop(columns=drop_cols, inplace=True)

print(f'✓ Final shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Final columns: {list(df.columns)}')
print()

## 3. Output & Summary

In [ ]:
# Show cleaned data preview
print('CLEANED DATA PREVIEW')
print('=' * 80)
print(df.head(5).to_string())
print()
print('Data types:')
print(df.dtypes)
print()
print('Missing values:')
print(df.isnull().sum())

## 4. Key Insights & Visualizations

In [ ]:
# Insight 1: Sales and Profit by Category
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

category_sales = df.groupby('category')['sale_price'].sum().sort_values(ascending=False)
category_profit = df.groupby('category')['profit'].sum().sort_values(ascending=False)

axes[0].bar(category_sales.index, category_sales.values, color='steelblue')
axes[0].set_title('Total Sales by Category', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Sale Price ($)')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(category_profit.index, category_profit.values, color='forestgreen')
axes[1].set_title('Total Profit by Category', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Profit ($)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('Sales and Profit Summary by Category:')
summary = pd.DataFrame({
    'Total Sales': category_sales,
    'Total Profit': category_profit,
    'Profit Margin %': (category_profit / category_sales * 100).round(2)
})
print(summary)

In [ ]:
# Insight 2: Segment Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

segment_count = df['segment'].value_counts()
segment_avg_profit = df.groupby('segment')['profit'].mean().sort_values(ascending=False)

axes[0].pie(segment_count.values, labels=segment_count.index, autopct='%1.1f%%', colors=['#ff9999', '#66b3ff', '#99ff99'])
axes[0].set_title('Order Distribution by Segment', fontsize=12, fontweight='bold')

axes[1].bar(segment_avg_profit.index, segment_avg_profit.values, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[1].set_title('Average Profit by Segment', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Avg Profit ($)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('Segment Breakdown:')
print(f'{segment_count}')

In [ ]:
# Insight 3: Discount vs Profit Relationship
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: Discount vs Profit
axes[0].scatter(df['discount'], df['profit'], alpha=0.5, s=20, color='purple')
axes[0].set_xlabel('Discount ($)')
axes[0].set_ylabel('Profit ($)')
axes[0].set_title('Discount vs Profit Relationship', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Quantity vs Profit
axes[1].scatter(df['quantity'], df['profit'], alpha=0.5, s=20, color='orange')
axes[1].set_xlabel('Quantity (units)')
axes[1].set_ylabel('Profit ($)')
axes[1].set_title('Quantity vs Profit Relationship', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Correlation Analysis:')
print(f'Discount-Profit correlation: {df[["discount", "profit"]].corr().iloc[0, 1]:.3f}')
print(f'Quantity-Profit correlation: {df[["quantity", "profit"]].corr().iloc[0, 1]:.3f}')

In [ ]:
# Insight 4: Sales Trend over Time
df_sorted = df.sort_values('order_date')
df_sorted['month'] = df_sorted['order_date'].dt.to_period('M')
monthly_sales = df_sorted.groupby('month').agg({
    'sale_price': 'sum',
    'profit': 'sum',
    'order_id': 'count'
}).rename(columns={'order_id': 'order_count'})

fig, ax = plt.subplots(figsize=(14, 5))
ax2 = ax.twinx()

ax.plot(range(len(monthly_sales)), monthly_sales['sale_price'].values, marker='o', color='steelblue', label='Sales', linewidth=2)
ax2.plot(range(len(monthly_sales)), monthly_sales['profit'].values, marker='s', color='forestgreen', label='Profit', linewidth=2)

ax.set_xlabel('Month')
ax.set_ylabel('Sales ($)', color='steelblue')
ax2.set_ylabel('Profit ($)', color='forestgreen')
ax.set_title('Monthly Sales and Profit Trend', fontsize=12, fontweight='bold')
ax.tick_params(axis='y', labelcolor='steelblue')
ax2.tick_params(axis='y', labelcolor='forestgreen')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Final Output

The cleaned and processed data is saved to `../outputs/processed_orders.csv`.

In [ ]:
# Save processed data (same as retail_analysis.py does)
output_path = '../outputs/processed_orders.csv'
df.to_csv(output_path, index=False)

print(f'✓ Processed data saved to: {output_path}')
print(f'  File size: {os.path.getsize(output_path) / 1024:.1f} KB')
print(f'  Rows: {len(df)}, Columns: {len(df.columns)}')
print()
print('Summary Statistics:')
print(df[['discount', 'sale_price', 'profit', 'quantity']].describe().round(2))

## 6. Next Steps

**From here, you can:**

1. **Upload to SQL Server** — run `python retail_analysis.py --upload-sql` to load into `dbo.df_orders`.
2. **Advanced Analysis** — pivot tables, cohort analysis, forecasting.
3. **Dashboard** — use Power BI, Tableau, or Plotly to visualize the `processed_orders.csv`.
4. **Automation** — schedule the ETL via cron or task scheduler.